# 02. Тензоры и матрицы в PyTorch

Это первый учебный блокнот проекта **Hands-on Transformer**. Здесь мы разберём, как хранить числа в PyTorch и выполнять операции, которые позже будут использоваться внутри Transformer.

## Что узнаем
1. Что такое переменная и тензор.
2. Чем отличаются скаляр, вектор и матрица.
3. Что означает `shape` — размерность массива.
4. Чем поэлементное умножение отличается от матричного.
5. Почему размеры матриц важны для Attention.

**Важно о выборе библиотеки.** В папке `sandbox` находится первоначальный прототип преподавателя на JAX. Мы сохраняем его без изменений. Основную учебную реализацию будем писать на PyTorch: он уже проверен в нашем Colab и используется в nanoGPT.

## 1. Несколько базовых терминов

- **Переменная** — имя, под которым программа хранит какое-то значение.
- **Библиотека** — готовый набор инструментов, который можно подключить к программе. PyTorch — библиотека для работы с тензорами и нейросетями.
- **Тензор (tensor)** — массив чисел. Он может иметь одно измерение, два измерения или больше.
- **Размерность (`shape`)** — размеры тензора по каждой оси. Например, матрица из 2 строк и 3 столбцов имеет `shape = (2, 3)`.
- **Ось (`axis`)** — одно из направлений массива. У матрицы ось 0 — строки, ось 1 — столбцы.

Скаляр — одно число; вектор — одномерный массив; матрица — двумерный массив.

In [ ]:
# Подключаем PyTorch и проверяем, что он доступен.
import torch

print("Версия PyTorch:", torch.__version__)

# Выбираем GPU, если он доступен, иначе используем CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство для вычислений:", device)

## 2. Скаляр, вектор и матрица

Создадим три тензора. Обрати внимание: функция `torch.tensor(...)` превращает переданные числа в тензор.

Пока не нужно запоминать все команды. Важно увидеть, как меняется `shape`.

In [ ]:
# Скаляр: одно число.
scalar = torch.tensor(7)

# Вектор: одна последовательность чисел.
vector = torch.tensor([10, 20, 30])

# Матрица: две строки и три столбца.
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

print("Скаляр:", scalar)
print("Форма скаляра:", scalar.shape, "Количество осей:", scalar.ndim)

print("\nВектор:", vector)
print("Форма вектора:", vector.shape, "Количество осей:", vector.ndim)

print("\nМатрица:\n", matrix)
print("Форма матрицы:", matrix.shape, "Количество осей:", matrix.ndim)

### Как читать результат?

Ожидаемые формы:
- скаляр: `torch.Size([])` — у него нет осей;
- вектор: `torch.Size([3])` — одна ось длиной 3;
- матрица: `torch.Size([2, 3])` — 2 строки и 3 столбца.

`ndim` показывает число осей. Это не то же самое, что количество элементов: в матрице `2 × 3` шесть элементов, но только две оси.

## 3. Операции с тензорами

Сначала сложим и перемножим два вектора одинаковой длины. Такие операции выполняются **поэлементно**: первое число с первым, второе со вторым и так далее.

In [ ]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([10, 20, 30])

print("a + b =", a + b)
print("a * b =", a * b)

Теперь главное для Transformer — **матричное умножение**.

Знак `@` в Python означает матричное умножение. Это не то же самое, что `*`.

Для умножения матрицы формы `(2, 3)` на матрицу формы `(3, 2)` внутренние размеры должны совпасть: `3` и `3`. Результат будет иметь форму `(2, 2)` — число строк первой матрицы и число столбцов второй.

In [ ]:
A = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])  # форма (2, 3)

B = torch.tensor([
    [10, 20],
    [30, 40],
    [50, 60]
])  # форма (3, 2)

C = A @ B

print("A имеет форму:", A.shape)
print("B имеет форму:", B.shape)
print("Результат A @ B:\n", C)
print("Форма результата:", C.shape)

### Откуда берётся первое число результата?

Первая строка `A` — `[1, 2, 3]`. Первый столбец `B` — `[10, 30, 50]`.

Их скалярное произведение:

`1 × 10 + 2 × 30 + 3 × 50 = 220`.

Поэтому элемент в первой строке и первом столбце результата равен 220. Остальные элементы считаются так же: строка первой матрицы умножается на столбец второй.

## 4. Как это связано с Transformer?

В Transformer для каждого токена сначала есть числовое представление — вектор эмбеддинга. Если в предложении 4 токена, а размерность каждого вектора равна 3, их можно представить матрицей формы `(4, 3)`.

Затем эту матрицу умножают на обучаемую матрицу проекции `W_Q`. Она преобразует каждое представление в Query-вектор.

В нашем маленьком примере:
- `X` имеет форму `(4, 3)` — 4 токена, по 3 числа на каждый;
- `W_Q` имеет форму `(3, 2)` — преобразует 3 числа в 2;
- `Q = X @ W_Q` имеет форму `(4, 2)`.

Пока не углубляемся в смысл Query. Сейчас важно понять, как размеры матриц определяют размер результата.

In [ ]:
X = torch.tensor([
    [1.0, 0.5, 0.2],
    [0.1, 0.7, 0.3],
    [0.4, 0.2, 0.9],
    [0.8, 0.1, 0.5]
])  # 4 токена, 3 числа на токен

W_Q = torch.tensor([
    [0.2, 0.4],
    [0.5, 0.1],
    [0.3, 0.6]
])  # преобразует 3 числа в 2

Q = X @ W_Q

print("Форма X:", X.shape)
print("Форма W_Q:", W_Q.shape)
print("Форма Q:", Q.shape)
print("Q:\n", Q)

## 5. Небольшое самостоятельное упражнение

Создай матрицу `M` формы `(2, 2)` и вектор `v` формы `(2,)`. Умножь матрицу на вектор через `@`.

Например, можешь взять:

`M = [[1, 2], [3, 4]]`

`v = [10, 100]`

**До запуска попробуй предсказать:** сколько чисел будет в результате?

In [ ]:
# Напиши здесь решение упражнения.
M = torch.tensor([
    [1, 2],
    [3, 4]
])

v = torch.tensor([10, 100])

# Умножь M на v и выведи результат и его форму.
# result = ...
# print(result)
# print(result.shape)

## Итог

Сегодня мы разобрали три идеи:
1. Тензор — это массив чисел; `shape` показывает его размеры.
2. `*` умножает элементы попарно, а `@` выполняет матричное умножение.
3. В Transformer матричные умножения преобразуют эмбеддинги в Query, Key и Value.

**Следующий шаг:** разобрать, как слова превращаются в числовые идентификаторы (токенизацию), а затем — как из идентификаторов получить эмбеддинги.